<a href="https://colab.research.google.com/github/mohamedalangr/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


*   *Lane Selection: Lane 2: Refresh / Content Opportunity Scoring.*

*   *ML Task Type: Supervised Binary Classification & Learning to Rank (Probability Scoring).*
*   *Explanation: The task is to predict the probability ($P(\text{decline} = 1 \mid X)$) that a specific content item will experience a performance decline based on historic, pre-decision feature signals. We then use these probabilistic outputs to rank pages into a prioritized review queue.*




## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


*   *Chosen Target/Proxy Label: is_declining_label (where trend_direction == "down").*
*   *Operational Definition: In this starter setup, the target is a binary indicator of whether a page's recent 90-day visibility trend is explicitly downward. In a production pipeline, this acts as a proxy for identifying high-exposure pages experiencing content decay or decay risk prior to absolute traffic loss.*



## 3. Success metric

*One metric you can defend. What number means 'good'?*


*   *Primary Success Metric: Precision@K (specifically Precision@50) and Average Precision (AP).*
*   *Rationale: Because this is a decision-support system for a human content team with limited capacity, global metrics like standard accuracy or ROC AUC can be misleading due to class imbalances. What matters most is how many of the top 50 pages prioritized by our model actually require a refresh. If our model achieves a high Precision@50, it directly minimizes wasted human editorial hours.*



## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd
import os

# Define robust paths to locate the starter data in Colab's virtual environment
paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv'
]

df = None
for path in paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Loaded from local path: {path}")
        break

if df is None:
    print("Fetching directly from source repository link...")
    url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    df = pd.read_csv(url)

# 1. Deduplicate by grain as required by the starter pipeline prep steps
df_clean = df.dropna(subset=['trend_direction']).copy()
df_clean = df_clean.drop_duplicates(subset=['content_id'])

# 2. Engineer the target column proxy live
df_clean['target_is_declining'] = (df_clean['trend_direction'] == 'down').astype(int)

print("\n--- DATA GRAIN & TARGET PREVIEW ---")
print(f"The Unit of Analysis (Grain): One Row = One Unique Content Item (content_id)")
print(f"Dataframe Shape: {df_clean.shape[0]} unique rows x {df_clean.shape[1]} columns\n")

# Display a clean view of the grain and engineered target column
preview_cols = ['content_id', 'client_id', 'impressions_90d', 'trend_direction', 'target_is_declining']
print(df_clean[preview_cols].head(5).to_string(index=False))

Fetching directly from source repository link...

--- DATA GRAIN & TARGET PREVIEW ---
The Unit of Analysis (Grain): One Row = One Unique Content Item (content_id)
Dataframe Shape: 30000 unique rows x 45 columns

          content_id         client_id  impressions_90d trend_direction  target_is_declining
content_304f48230142 client_f369cb89fc             3803            down                    1
content_a1fb4e703a9e client_4e07408562            15320            down                    1
content_9aa793d4d895 client_7f2253d7e2            12581            down                    1
content_331d6c4de07b client_19581e27de            11751          stable                    0
content_d99b7a2d90ca client_3fdba35f04            19140            down                    1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

*A human-written, static rule (e.g., “flag a page if impressions > 500 and days since update > 180”) treats all variables with independent, linear thresholds. It cannot account for the highly complex, non-linear interactions between variables—such as how a lower word count interacts with seasonal search volume or position shifts across various intent tiers. Machine Learning learns a dynamic boundary across all signals simultaneously, allowing it to rank pages based on fine-grained probability curves rather than rigid, binary cutoffs.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.